# MACD (Moving Average Convergence Divergence) - Complete Trading Tutorial

## Overview
This notebook provides a comprehensive tutorial on the MACD indicator, including:
- Concept and theory
- Mathematical formula and calculation
- Manual implementation in Python
- TradingView integration
- Binance platform usage
- Practical trading strategies
- Backtesting examples

## 1. Introduction to MACD

### What is MACD?
The Moving Average Convergence Divergence (MACD) is a trend-following momentum indicator that shows the relationship between two exponential moving averages (EMAs) of a security's price. It was developed by Gerald Appel in the late 1970s.

### Why Use MACD?
- **Trend Identification**: Excellent for identifying trend direction and strength
- **Momentum Analysis**: Shows both trend and momentum in one indicator
- **Signal Generation**: Provides clear buy/sell signals
- **Divergence Detection**: Helps identify potential trend reversals
- **Versatility**: Works well in various market conditions

### Best Timeframes and Markets
- **Short-term**: 5-15 minute charts for day trading
- **Medium-term**: 1-4 hour charts for swing trading
- **Long-term**: Daily charts for position trading
- **Markets**: Works well in trending markets, also useful for ranging markets

## 2. Mathematical Formula

### MACD Components
```
MACD Line = 12-period EMA - 26-period EMA
Signal Line = 9-period EMA of MACD Line
Histogram = MACD Line - Signal Line
```

### Parameters
- **Fast EMA Period**: Typically 12 periods
- **Slow EMA Period**: Typically 26 periods
- **Signal Line Period**: Typically 9 periods
- **Histogram**: The difference between MACD line and Signal line

### Interpretation
- **MACD Line Above Signal Line**: Bullish momentum
- **MACD Line Below Signal Line**: Bearish momentum
- **Histogram Above Zero**: Bullish momentum strengthening
- **Histogram Below Zero**: Bearish momentum strengthening

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ccxt
from datetime import datetime, timedelta

# Import utilities
from utils.data_downloader import DataDownloader
from utils.indicators import IndicatorCalculator
from utils.backtest_engine import BacktestEngine, StrategyGenerator

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")

## 3. Data Download

Let's download historical crypto data for our analysis.

In [ ]:
# Initialize data downloader
downloader = DataDownloader()

# Download BTC/USDT data
pair = "BTC/USDT"
timeframe = "1d"
days = 365

print(f"Downloading {pair} data for {days} days...")
df = downloader.get_market_data(pair, timeframe, days, source='ccxt')

if df is not None:
    print(f"\nDownloaded {len(df)} candles of {pair} data")
    print(f"Date range: {df.index[0]} to {df.index[-1]}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print("Failed to download data")

## 4. Manual Calculation of MACD

Let's implement MACD from scratch to understand how it works.

In [ ]:
def calculate_ema(data, period):
    """
    Calculate Exponential Moving Average
    """
    ema = np.zeros(len(data))
    multiplier = 2 / (period + 1)
    
    # Initialize with SMA
    sma = data.iloc[:period].mean()
    ema[period - 1] = sma
    
    # Calculate EMA for remaining values
    for i in range(period, len(data)):
        ema[i] = (data.iloc[i] * multiplier) + (ema[i-1] * (1 - multiplier))
    
    return pd.Series(ema, index=data.index)

def calculate_macd_manual(data, fast_period=12, slow_period=26, signal_period=9):
    """
    Calculate MACD manually
    
    Parameters:
    - data: pandas Series with price data
    - fast_period: Fast EMA period (default 12)
    - slow_period: Slow EMA period (default 26)
    - signal_period: Signal line period (default 9)
    
    Returns:
    - DataFrame with MACD components
    """
    df = pd.DataFrame(index=data.index)
    
    # Calculate EMAs
    df['ema_fast'] = calculate_ema(data, fast_period)
    df['ema_slow'] = calculate_ema(data, slow_period)
    
    # Calculate MACD line
    df['macd_line'] = df['ema_fast'] - df['ema_slow']
    
    # Calculate signal line
    df['signal_line'] = calculate_ema(df['macd_line'], signal_period)
    
    # Calculate histogram
    df['histogram'] = df['macd_line'] - df['signal_line']
    
    return df

# Calculate MACD
df_macd = calculate_macd_manual(df['close'])

print("MACD calculated manually:")
display(df_macd.tail(10))

# Let's verify the calculation manually
print("\nManual verification of MACD for the last row:")
last_ema_fast = df_macd['ema_fast'].iloc[-1]
last_ema_slow = df_macd['ema_slow'].iloc[-1]
last_macd = df_macd['macd_line'].iloc[-1]
last_signal = df_macd['signal_line'].iloc[-1]
last_histogram = df_macd['histogram'].iloc[-1]

print(f"Last EMA(12): {last_ema_fast:.2f}")
print(f"Last EMA(26): {last_ema_slow:.2f}")
print(f"MACD Line: {last_macd:.2f}")
print(f"Signal Line: {last_signal:.2f}")
print(f"Histogram: {last_histogram:.2f}")
print(f"Verification: {last_macd:.2f} - {last_signal:.2f} = {last_histogram:.2f}")

## 5. Library Implementation

Now let's use established libraries to calculate MACD.

In [ ]:
# Using our utilities
df_utils = df.copy()
macd_utils = IndicatorCalculator.macd(df_utils['close'], 12, 26, 9)
df_utils['macd_line'] = macd_utils['macd']
df_utils['signal_line'] = macd_utils['signal']
df_utils['histogram'] = macd_utils['histogram']

# Using pandas-ta
import pandas_ta as ta
df_ta = df.copy()
df_ta.ta.macd(fast=12, slow=26, signal=9, append=True)

# Using TA-Lib
try:
    import talib
    df_talib = df.copy()
    df_talib['macd_line'], df_talib['signal_line'], df_talib['histogram'] = talib.MACD(
        df_talib['close'].values, fastperiod=12, slowperiod=26, signalperiod=9
    )
    print("TA-Lib MACD calculated successfully")
except ImportError:
    print("TA-Lib not installed, skipping TA-Lib example")
    df_talib = df.copy()
    df_talib['macd_line'] = np.nan
    df_talib['signal_line'] = np.nan
    df_talib['histogram'] = np.nan

# Compare results
print("Comparison of different MACD implementations:")
comparison = pd.DataFrame({
    'Manual_MACD': df_macd['macd_line'],
    'Utils_MACD': df_utils['macd_line'],
    'Pandas-TA_MACD': df_ta['MACD_12_26_9'],
    'TA-Lib_MACD': df_talib['macd_line']
})

display(comparison.dropna().tail(10))

# Check if all methods produce similar results
print("\nDifferences between methods:")
print(f"Manual vs Utils max difference: {abs(comparison['Manual_MACD'] - comparison['Utils_MACD']).max():.8f}")
print(f"Manual vs Pandas-TA max difference: {abs(comparison['Manual_MACD'] - comparison['Pandas-TA_MACD']).max():.8f}")

## 6. Visualization

Let's visualize MACD with price action.

In [ ]:
# Create interactive plot
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(f'{pair} - Price Action', f'{pair} - MACD Indicator'),
    vertical_spacing=0.1,
    row_heights=[0.7, 0.3]
)

# Price chart
fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='Price'
    ),
    row=1, col=1
)

# MACD line
fig.add_trace(
    go.Scatter(
        x=df_macd.index,
        y=df_macd['macd_line'],
        mode='lines',
        name='MACD Line',
        line=dict(color='blue', width=2)
    ),
    row=2, col=1
)

# Signal line
fig.add_trace(
    go.Scatter(
        x=df_macd.index,
        y=df_macd['signal_line'],
        mode='lines',
        name='Signal Line',
        line=dict(color='red', width=2)
    ),
    row=2, col=1
)

# Histogram
colors = ['green' if x >= 0 else 'red' for x in df_macd['histogram']]
fig.add_trace(
    go.Bar(
        x=df_macd.index,
        y=df_macd['histogram'],
        name='Histogram',
        marker_color=colors,
        opacity=0.7
    ),
    row=2, col=1
)

# Add zero line
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    title=f'{pair} - MACD Analysis',
    template='plotly_dark',
    height=800,
    xaxis_rangeslider_visible=False
)

fig.show()

# Create static plot
plt.figure(figsize=(15, 10))

# Price chart
plt.subplot(2, 1, 1)
plt.plot(df.index, df['close'], label='Price', color='black', linewidth=1)
plt.title(f'{pair} - Price Action')
plt.ylabel('Price (USDT)')
plt.legend()
plt.grid(True, alpha=0.3)

# MACD chart
plt.subplot(2, 1, 2)
plt.plot(df_macd.index, df_macd['macd_line'], label='MACD Line', color='blue', linewidth=2)
plt.plot(df_macd.index, df_macd['signal_line'], label='Signal Line', color='red', linewidth=2)
plt.bar(df_macd.index, df_macd['histogram'], label='Histogram', 
        color=['green' if x >= 0 else 'red' for x in df_macd['histogram']], alpha=0.7)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.title('MACD Indicator')
plt.ylabel('MACD Value')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. TradingView Integration

### Pine Script for MACD

```pine
// MACD Pine Script
//@version=5
indicator("MACD", shorttitle="MACD", overlay=false)

// Input parameters
fastLength = input.int(12, "Fast Length", minval=1)
slowLength = input.int(26, "Slow Length", minval=1)
signalLength = input.int(9, "Signal Length", minval=1)

// Calculate MACD
[macdLine, signalLine, histLine] = ta.macd(close, fastLength, slowLength, signalLength)

// Plot MACD line
plot(macdLine, "MACD Line", color=color.blue, linewidth=2)
plot(signalLine, "Signal Line", color=color.red, linewidth=2)

// Plot histogram
plot(histLine, "Histogram", style=plot.style_columns, color=histLine >= 0 ? color.green : color.red)

// Plot zero line
hline(0, "Zero Line", color=color.gray, linestyle=hline.style_dashed)

// Add crossover signals
buySignal = ta.crossover(macdLine, signalLine)
sellSignal = ta.crossunder(macdLine, signalLine)

plotshape(buySignal, title="Buy Signal", location=location.bottom, color=color.green, style=shape.triangleup, size=size.small)
plotshape(sellSignal, title="Sell Signal", location=location.top, color=color.red, style=shape.triangledown, size=size.small)

// Add divergence detection (optional)
bullishDiv = ta.crossunder(macdLine, signalLine) and macdLine < 0
bearishDiv = ta.crossover(macdLine, signalLine) and macdLine > 0

plotshape(bullishDiv, title="Bullish Divergence", location=location.bottom, color=color.lime, style=shape.circle, size=size.tiny)
plotshape(bearishDiv, title="Bearish Divergence", location=location.top, color=color.maroon, style=shape.circle, size=size.tiny)
```

### How to Add to TradingView:
1. Open TradingView chart
2. Click on "Pine Editor" tab at the bottom
3. Copy and paste the script above
4. Click "Add to Chart"
5. Adjust parameters in the settings

### TradingView Built-in MACD:
1. Click on "Indicators" at the top
2. Search for "MACD"
3. Select "Moving Average Convergence Divergence"
4. Adjust parameters as needed

## 8. Binance Integration

### How to Add MACD on Binance:
1. Open Binance trading interface
2. Select your trading pair (e.g., BTC/USDT)
3. Click on "Indicators" at the top of the chart
4. Search for "MACD"
5. Select "MACD" from the list
6. Adjust parameters in the settings
7. Click "Apply"

### Binance MACD Parameters:
- **Fast Length**: Typically 12 periods
- **Slow Length**: Typically 26 periods
- **Signal Length**: Typically 9 periods
- **Source**: Close price (default)
- **Colors**: Customizable for MACD line, signal line, and histogram

## 9. Trading Strategies

### Strategy 1: MACD Crossover
**Description**: Buy when MACD line crosses above signal line, sell when it crosses below

**Rules**:
- **Entry**: MACD line crosses above signal line
- **Exit**: MACD line crosses below signal line
- **Stop Loss**: 2-3% below entry price
- **Take Profit**: 5-10% above entry price or when opposite signal occurs

In [ ]:
def macd_crossover_strategy(data, fast_period=12, slow_period=26, signal_period=9):
    """
    MACD Crossover Strategy
    """
    df = data.copy()
    
    # Calculate MACD
    macd_data = calculate_macd_manual(df['close'], fast_period, slow_period, signal_period)
    df['macd_line'] = macd_data['macd_line']
    df['signal_line'] = macd_data['signal_line']
    df['histogram'] = macd_data['histogram']
    
    # Generate signals
    df['signal'] = 0
    df.loc[df['macd_line'] > df['signal_line'], 'signal'] = 1  # Long position
    df.loc[df['macd_line'] < df['signal_line'], 'signal'] = -1  # Short position
    
    # Identify crossovers
    df['crossover'] = ((df['macd_line'] > df['signal_line']) & 
                       (df['macd_line'].shift(1) <= df['signal_line'].shift(1)))
    df['crossunder'] = ((df['macd_line'] < df['signal_line']) & 
                        (df['macd_line'].shift(1) >= df['signal_line'].shift(1)))
    
    return df

# Apply strategy
df_signals = macd_crossover_strategy(df)

print(f"Total buy signals: {df_signals['crossover'].sum()}")
print(f"Total sell signals: {df_signals['crossunder'].sum()}")
print(f"\nRecent signals:")
recent_signals = df_signals[df_signals['crossover'] | df_signals['crossunder']].tail(10)
display(recent_signals[['close', 'macd_line', 'signal_line', 'crossover', 'crossunder']])

### Strategy 2: MACD Divergence
**Description**: Trade based on divergence between price and MACD

**Rules**:
- **Bullish Divergence**: Price makes lower lows while MACD makes higher lows
- **Bearish Divergence**: Price makes higher highs while MACD makes lower highs
- **Entry**: When divergence is confirmed
- **Exit**: When opposite signal occurs or after reaching profit target

In [ ]:
def find_divergence(data, window=5):
    """
    Find MACD divergences
    """
    df = data.copy()
    df['bullish_divergence'] = False
    df['bearish_divergence'] = False
    
    for i in range(window, len(df)):
        # Bullish divergence: price lower lows, MACD higher lows
        price_window = df['close'].iloc[i-window:i+1]
        macd_window = df['macd_line'].iloc[i-window:i+1]
        
        if len(price_window) >= 3:
            price_low_idx = price_window.idxmin()
            macd_low_idx = macd_window.idxmin()
            
            # Check if we have a divergence pattern
            if (price_low_idx < macd_low_idx and 
                price_window.min() < df['close'].iloc[i-window-1] and
                macd_window.min() > df['macd_line'].iloc[i-window-1]):
                df.loc[df.index[i], 'bullish_divergence'] = True
        
        # Bearish divergence: price higher highs, MACD lower highs
        price_high_idx = price_window.idxmax()
        macd_high_idx = macd_window.idxmax()
        
        if len(price_window) >= 3:
            if (price_high_idx < macd_high_idx and 
                price_window.max() > df['close'].iloc[i-window-1] and
                macd_window.max() < df['macd_line'].iloc[i-window-1]):
                df.loc[df.index[i], 'bearish_divergence'] = True
    
    return df

# Apply divergence detection
df_divergence = find_divergence(df_signals)

print(f"Bullish divergences found: {df_divergence['bullish_divergence'].sum()}")
print(f"Bearish divergences found: {df_divergence['bearish_divergence'].sum()}")

# Show divergences
divergences = df_divergence[df_divergence['bullish_divergence'] | df_divergence['bearish_divergence']]
if len(divergences) > 0:
    print("\nRecent divergences:")
    display(divergences[['close', 'macd_line', 'signal_line', 'bullish_divergence', 'bearish_divergence']].tail(10))

### Strategy 3: MACD Zero Line Crossover
**Description**: Trade based on MACD line crossing above/below zero line

**Rules**:
- **Entry**: MACD line crosses above zero line (bullish) or below zero line (bearish)
- **Exit**: When MACD line crosses back through zero line or when profit target reached
- **Stop Loss**: 2-3% beyond entry price
- **Take Profit**: 5-15% depending on market conditions

In [ ]:
def macd_zero_line_strategy(data, fast_period=12, slow_period=26, signal_period=9):
    """
    MACD Zero Line Crossover Strategy
    """
    df = data.copy()
    
    # Calculate MACD
    macd_data = calculate_macd_manual(df['close'], fast_period, slow_period, signal_period)
    df['macd_line'] = macd_data['macd_line']
    df['signal_line'] = macd_data['signal_line']
    df['histogram'] = macd_data['histogram']
    
    # Generate signals
    df['signal'] = 0
    df.loc[df['macd_line'] > 0, 'signal'] = 1  # Long position
    df.loc[df['macd_line'] < 0, 'signal'] = -1  # Short position
    
    # Identify zero line crossovers
    df['zero_crossover_up'] = ((df['macd_line'] > 0) & (df['macd_line'].shift(1) <= 0))
    df['zero_crossover_down'] = ((df['macd_line'] < 0) & (df['macd_line'].shift(1) >= 0))
    
    return df

# Apply strategy
df_zero_line = macd_zero_line_strategy(df)

print(f"Zero line crossover up signals: {df_zero_line['zero_crossover_up'].sum()}")
print(f"Zero line crossover down signals: {df_zero_line['zero_crossover_down'].sum()}")

# Create signals DataFrame for backtesting
zero_line_signals = pd.DataFrame(index=df_zero_line.index)
zero_line_signals['signal'] = df_zero_line['signal']

## 10. Backtesting Results

Let's backtest our strategies and analyze the results.

In [ ]:
# Initialize backtest engine
engine = BacktestEngine(initial_capital=10000)

# Create signals for each strategy
macd_crossover_signals = pd.DataFrame(index=df_signals.index)
macd_crossover_signals['signal'] = df_signals['signal']

zero_line_signals = pd.DataFrame(index=df_zero_line.index)
zero_line_signals['signal'] = df_zero_line['signal']

strategies = {
    'MACD_Crossover': macd_crossover_signals,
    'MACD_Zero_Line': zero_line_signals
}

# Run backtests
for name, signals in strategies.items():
    print(f"Backtesting {name} strategy...")
    engine.run_backtest(df, signals, name)

# Compare strategies
comparison = engine.compare_strategies(list(strategies.keys()))
print("\nStrategy Comparison:")
display(comparison)

# Plot equity curves
engine.plot_equity_curve()

# Plot drawdowns
engine.plot_drawdown()

## 11. Pros and Cons

### Advantages:
- ✅ **Trend and Momentum**: Combines trend following with momentum analysis
- ✅ **Clear Signals**: Provides unambiguous buy/sell signals
- ✅ **Divergence Detection**: Helps identify potential trend reversals
- ✅ **Versatile**: Works well in various market conditions
- ✅ **Widely Used**: Many traders watch MACD signals

### Disadvantages:
- ❌ **Lagging Indicator**: Based on past prices, signals are delayed
- ❌ **Whipsaw in Ranging Markets**: Can generate false signals in sideways markets
- ❌ **Complex**: Multiple components to interpret
- ❌ **False Signals**: Can give false signals during strong trends

### When to Use:
- **Trending Markets**: Strong uptrends or downtrends
- **Medium-term Trading**: Swing trading and position trading
- **Confirmation**: To confirm other indicators or signals
- **Divergence Trading**: When looking for trend reversals

### When to Avoid:
- **Ranging Markets**: Sideways or choppy markets
- **Very Short-term**: Scalping where lag is problematic
- **High Volatility**: During news events or high volatility
- **As Standalone**: Should not be used as the only indicator

## 12. Advanced MACD Techniques

### 1. MACD with Different Parameters
Use different parameter combinations for different market conditions:
- **Short-term**: 5, 35, 5 (faster signals)
- **Standard**: 12, 26, 9 (balanced signals)
- **Long-term**: 19, 39, 9 (smoother signals)

### 2. MACD Histogram Analysis
Monitor histogram changes for early signals:
- **Histogram Contraction**: Momentum slowing
- **Histogram Expansion**: Momentum increasing
- **Histogram Divergence**: Leading indicator of trend change

### 3. Multiple Time Frame Analysis
Use MACD on multiple timeframes for confirmation:
- **Long-term**: Daily chart for trend direction
- **Medium-term**: 4-hour chart for timing
- **Short-term**: 1-hour chart for entry/exit

In [ ]:
# Advanced MACD Techniques
df_advanced = df.copy()

# 1. MACD with different parameters
macd_standard = calculate_macd_manual(df_advanced['close'], 12, 26, 9)
macd_short_term = calculate_macd_manual(df_advanced['close'], 5, 35, 5)
macd_long_term = calculate_macd_manual(df_advanced['close'], 19, 39, 9)

df_advanced['macd_standard'] = macd_standard['macd_line']
df_advanced['macd_short_term'] = macd_short_term['macd_line']
df_advanced['macd_long_term'] = macd_long_term['macd_line']

# 2. MACD Histogram analysis
df_advanced['histogram_standard'] = macd_standard['histogram']
df_advanced['histogram_change'] = df_advanced['histogram_standard'].diff()
df_advanced['histogram_acceleration'] = df_advanced['histogram_change'].diff()

# 3. Signal strength based on histogram
df_advanced['signal_strength'] = abs(df_advanced['histogram_standard'])
df_advanced['strong_signal'] = df_advanced['signal_strength'] > df_advanced['signal_strength'].quantile(0.7)

# Plot advanced techniques
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Different parameter comparison
axes[0, 0].plot(df_advanced.index, df_advanced['macd_standard'], label='Standard (12,26,9)', color='blue')
axes[0, 0].plot(df_advanced.index, df_advanced['macd_short_term'], label='Short-term (5,35,5)', color='green')
axes[0, 0].plot(df_advanced.index, df_advanced['macd_long_term'], label='Long-term (19,39,9)', color='red')
axes[0, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 0].set_title('MACD with Different Parameters')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Histogram analysis
axes[0, 1].plot(df_advanced.index, df_advanced['histogram_standard'], label='Histogram', color='purple')
axes[0, 1].plot(df_advanced.index, df_advanced['histogram_change'], label='Histogram Change', color='orange')
axes[0, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Histogram Analysis')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Signal strength
axes[1, 0].plot(df_advanced.index, df_advanced['signal_strength'], label='Signal Strength', color='brown')
axes[1, 0].fill_between(df_advanced.index, df_advanced['signal_strength'], 
                        alpha=0.3, color='brown')
axes[1, 0].axhline(y=df_advanced['signal_strength'].quantile(0.7), color='red', 
                 linestyle='--', label='Strong Signal Threshold')
axes[1, 0].set_title('Signal Strength Analysis')
axes[1, 0].set_ylabel('Strength')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Signal strength distribution
axes[1, 1].hist(df_advanced['signal_strength'].dropna(), bins=30, alpha=0.7, color='brown', edgecolor='black')
axes[1, 1].axvline(x=df_advanced['signal_strength'].quantile(0.7), color='red', 
                 linestyle='--', label='70th percentile')
axes[1, 1].set_title('Signal Strength Distribution')
axes[1, 1].set_xlabel('Signal Strength')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analysis of strong signals
print("Strong Signal Analysis:")
strong_signals = df_advanced[df_advanced['strong_signal']]
print(f"Total strong signals: {len(strong_signals)} ({len(strong_signals)/len(df_advanced)*100:.1f}%)")
print(f"Average signal strength: {df_advanced['signal_strength'].mean():.4f}")
print(f"Strong signal threshold: {df_advanced['signal_strength'].quantile(0.7):.4f}")

## 13. Risk Management with MACD

### 1. Stop Loss Placement
- **Trend Following**: Place stop loss below recent swing low
- **Momentum Reversal**: Place stop loss beyond the signal point
- **Volatility Adjusted**: Use ATR to adjust stop loss distance

### 2. Position Sizing
- **Signal Strength**: Increase position size for strong signals
- **Market Context**: Reduce size during high volatility
- **Confirmation**: Increase size with multiple confirmations

### 3. Risk-Reward Ratio
- **Minimum Ratio**: 1:2 or better
- **Target Based**: Use recent price swings for targets
- **Trailing Stops**: Use trailing stops for trend following

In [ ]:
# Risk Management with MACD
df_risk = df.copy()

# Calculate MACD and other indicators
macd_data = calculate_macd_manual(df_risk['close'])
df_risk['macd_line'] = macd_data['macd_line']
df_risk['signal_line'] = macd_data['signal_line']
df_risk['histogram'] = macd_data['histogram']
df_risk['atr'] = IndicatorCalculator.atr(df_risk['high'], df_risk['low'], df_risk['close'], 14)

# 1. Stop loss based on ATR
stop_loss_multiplier = 2.0
df_risk['stop_loss_long'] = df_risk['close'] - (stop_loss_multiplier * df_risk['atr'])
df_risk['stop_loss_short'] = df_risk['close'] + (stop_loss_multiplier * df_risk['atr'])

# 2. Position sizing based on MACD signal strength
df_risk['signal_strength'] = abs(df_risk['histogram'])
df_risk['position_size'] = np.where(
    df_risk['signal_strength'] > df_risk['signal_strength'].quantile(0.8), 1.0,  # Large position
    np.where(
        df_risk['signal_strength'] > df_risk['signal_strength'].quantile(0.5), 0.5,  # Medium position
        0.25  # Small position
    )
)

# 3. Risk-reward ratio
df_risk['risk_per_unit'] = df_risk['atr'] * stop_loss_multiplier
df_risk['reward_per_unit'] = df_risk['atr'] * 3  # 3:1 reward-risk ratio
df_risk['risk_reward_ratio'] = df_risk['reward_per_unit'] / df_risk['risk_per_unit']

# Plot risk management
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Stop loss levels
axes[0, 0].plot(df_risk.index, df_risk['close'], label='Price', alpha=0.7, color='black')
axes[0, 0].plot(df_risk.index, df_risk['stop_loss_long'], label='Stop Loss (Long)', color='red', linestyle='--')
axes[0, 0].plot(df_risk.index, df_risk['stop_loss_short'], label='Stop Loss (Short)', color='green', linestyle='--')
axes[0, 0].set_title('Stop Loss Levels Based on ATR')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Position sizing
axes[0, 1].plot(df_risk.index, df_risk['position_size'], label='Position Size', color='purple', linewidth=2)
axes[0, 1].fill_between(df_risk.index, df_risk['position_size'], alpha=0.3, color='purple')
axes[0, 1].set_title('Position Sizing Based on MACD Signal Strength')
axes[0, 1].set_ylabel('Position Size (Multiplier)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Risk-reward ratio
axes[1, 0].plot(df_risk.index, df_risk['risk_reward_ratio'], label='Risk-Reward Ratio', color='orange')
axes[1, 0].axhline(y=3, color='red', linestyle='--', label='Minimum 3:1 Ratio')
axes[1, 0].fill_between(df_risk.index, df_risk['risk_reward_ratio'], 3, alpha=0.3, color='orange')
axes[1, 0].set_title('Risk-Reward Ratio Analysis')
axes[1, 0].set_ylabel('Ratio')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Signal strength distribution
axes[1, 1].hist(df_risk['signal_strength'].dropna(), bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[1, 1].axvline(x=df_risk['signal_strength'].quantile(0.5), color='red', linestyle='--', label='50th percentile')
axes[1, 1].axvline(x=df_risk['signal_strength'].quantile(0.8), color='green', linestyle='--', label='80th percentile')
axes[1, 1].set_title('MACD Signal Strength Distribution')
axes[1, 1].set_xlabel('Signal Strength')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Risk management statistics
print("Risk Management Statistics:")
print(f"Average position size: {df_risk['position_size'].mean():.2f}")
print(f"Average risk-reward ratio: {df_risk['risk_reward_ratio'].mean():.2f}")
print(f"Trades with minimum 3:1 ratio: {(df_risk['risk_reward_ratio'] >= 3).sum()} ({(df_risk['risk_reward_ratio'] >= 3).mean()*100:.1f}%)")
print(f"Average stop loss distance: {df_risk['risk_per_unit'].mean():.2f}")

## 14. Conclusion

The Moving Average Convergence Divergence (MACD) is one of the most versatile and widely used technical indicators. It combines trend-following and momentum analysis in a single indicator, making it valuable for traders of all experience levels.

### Key Takeaways:
1. **Trend and Momentum**: MACD excels at identifying both trend direction and momentum
2. **Clear Signals**: Provides unambiguous buy/sell signals through crossovers
3. **Divergence Detection**: Helps identify potential trend reversals early
4. **Risk Management**: Can be used for position sizing and stop loss placement
5. **Combination**: Works best when combined with other indicators

### Best Practices:
- Use multiple parameter combinations for different market conditions
- Combine with support/resistance levels for confirmation
- Implement proper risk management techniques
- Consider market context when interpreting signals
- Test different strategies to find what works best for your trading style

## 15. Exercises

### Exercise 1: Optimize MACD Parameters
Test different MACD parameter combinations to find the optimal settings for crossover strategies. Create a function that tests multiple parameter combinations and returns the best performing one based on Sharpe ratio.

### Exercise 2: Create MACD Strategy with RSI Confirmation
Modify the MACD crossover strategy to include RSI confirmation. Only enter trades when RSI confirms the MACD signal (e.g., RSI above 50 for bullish signals, below 50 for bearish signals).

### Exercise 3: Implement MACD Divergence Trading System
Create a comprehensive divergence trading system that identifies and trades both bullish and bearish divergences with proper risk management.

### Solutions:
```python
# Exercise 1 Solution
def optimize_macd_parameters(data, fast_range, slow_range, signal_range):
    best_sharpe = -float('inf')
    best_params = None
    
    for fast in fast_range:
        for slow in slow_range:
            for signal in signal_range:
                if fast >= slow:
                    continue
                
                signals = macd_crossover_strategy(data, fast, slow, signal)[['signal']]
                engine = BacktestEngine()
                result = engine.run_backtest(data, signals, f"MACD_{fast}_{slow}_{signal}")
                sharpe = result['metrics']['sharpe_ratio']
                
                if sharpe > best_sharpe:
                    best_sharpe = sharpe
                    best_params = (fast, slow, signal)
    
    return best_params, best_sharpe

# Exercise 2 Solution
def macd_rsi_strategy(data, rsi_period=14, rsi_threshold=50):
    df = data.copy()
    
    # Calculate MACD
    macd_data = calculate_macd_manual(df['close'])
    df['macd_line'] = macd_data['macd_line']
    df['signal_line'] = macd_data['signal_line']
    
    # Calculate RSI
    df['rsi'] = IndicatorCalculator.rsi(df['close'], rsi_period)
    
    # Generate signals with RSI confirmation
    df['signal'] = 0
    
    # Buy signal: MACD crossover AND RSI above threshold
    buy_condition = ((df['macd_line'] > df['signal_line']) & 
                    (df['macd_line'].shift(1) <= df['signal_line'].shift(1)) &
                    (df['rsi'] > rsi_threshold))
    
    # Sell signal: MACD crossunder AND RSI below threshold
    sell_condition = ((df['macd_line'] < df['signal_line']) & 
                      (df['macd_line'].shift(1) >= df['signal_line'].shift(1)) &
                      (df['rsi'] < (100 - rsi_threshold)))
    
    df.loc[buy_condition, 'signal'] = 1
    df.loc[sell_condition, 'signal'] = -1
    
    return df
```

## 16. Additional Resources

- **Books**:
  - "Technical Analysis of Financial Markets" by John J. Murphy
  - "Trading Systems and Methods" by Perry J. Kaufman
  - "Come Into My Trading Room" by Dr. Alexander Elder
  - "MACD: The Ultimate Guide" by Gerald Appel

- **Online Resources**:
  - [TradingView Documentation](https://www.tradingview.com/pine-script-docs/)
  - [Investopedia - MACD](https://www.investopedia.com/terms/m/macd.asp)
  - [Binance Academy - Technical Analysis](https://academy.binance.com/en/categories/technical-analysis)

- **Research Papers**:
  - "The Profitability of Moving Average Trading Rules" by Brock, Lakonishok, and LeBaron
  - "Technical Analysis and Moving Averages" by Lo, Mamaysky, and Wang

- **Tools**:
  - [TradingView](https://www.tradingview.com/) - Charting and Pine Script
  - [Binance](https://www.binance.com/) - Trading and indicators
  - [Freqtrade](https://www.freqtrade.io/) - Algorithmic trading

---

**Disclaimer**: This tutorial is for educational purposes only. Trading cryptocurrencies involves significant risk. Always do your own research and consider consulting with a financial advisor before making investment decisions.